# Training Setup — RunPod
Works on **RunPod** (primary target) and **local Windows/Linux**. Run cells top-to-bottom. Setup cells (1–8) only need to run once per pod.

**RunPod tips**
- Mount a **persistent network volume** at `/workspace` so checkpoints and dataset survive pod restarts.
- Recommended pod: PyTorch 2.1 / CUDA 11.8 image, A4000/A5000/A6000 or 4090.
- The training cell streams output live and saves checkpoints to `models/last.pt` + `models/best.pt`. If the pod dies, restart and re-run cells 0–10, then set `RESUME = "models/last.pt"` in cell 10 to continue.

## 0. Detect platform

In [1]:
import platform, sys, os

IS_WINDOWS = platform.system() == "Windows"
IS_RUNPOD  = os.path.exists("/workspace")

REPO_NAME = "Pattern_Recog_Project"
REPO_URL  = "https://github.com/RATSAPORN/Pattern_Recog_Project.git"  # change if your fork lives elsewhere

if IS_RUNPOD:
    PROJECT = f"/workspace/{REPO_NAME}"
else:
    PROJECT = os.path.abspath(os.path.join(os.getcwd(), ".."))

print(f"Platform : {platform.system()}")
print(f"RunPod   : {IS_RUNPOD}")
print(f"Project  : {PROJECT}")

Platform : Linux
RunPod   : True
Project  : /workspace/Pattern_Recog_Project


## 1. Clone or pull latest code (RunPod only)
On a fresh pod, this clones the repo into `/workspace`. On subsequent runs it just pulls the latest commits.

In [2]:
import subprocess
if IS_RUNPOD:
    if not os.path.exists(PROJECT):
        print(f"Cloning {REPO_URL} into {PROJECT}")
        subprocess.run(["git", "clone", REPO_URL, PROJECT], check=True)
    else:
        print(f"Pulling latest in {PROJECT}")
        subprocess.run(["git", "-C", PROJECT, "pull"], check=True)
else:
    print("Local machine — skipping git pull. Make sure your code is up to date.")

Pulling latest in /workspace/Pattern_Recog_Project
Already up to date.


## 2. Java for METEOR/CIDEr metrics
- **RunPod/Linux**: installs automatically
- **Windows**: install manually from https://adoptium.net (Temurin JRE 17)

In [3]:
import shutil
if IS_WINDOWS:
    if shutil.which("java"):
        print("Java found:", shutil.which("java"))
    else:
        print("Java not found. Install from https://adoptium.net then restart the notebook.")
else:
    os.system("apt-get update -q && apt-get install -y --no-install-recommends default-jre-headless -q")
    os.system("java -version")

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1601 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4192 kB]
Fetched 6049 kB in 3s (1750 kB/s)
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
default-jre-headless is already the newest version (2:1.11-72build2).
0 upgraded, 0 newly installed, 0 to remove and 168 not upgraded.


openjdk version "11.0.30" 2026-01-20
OpenJDK Runtime Environment (build 11.0.30+7-post-Ubuntu-1ubuntu122.04)
OpenJDK 64-Bit Server VM (build 11.0.30+7-post-Ubuntu-1ubuntu122.04, mixed mode, sharing)


## 3. PyTorch
- **RunPod** (CUDA 11.8): pins `torch==2.1.0+cu118`
- **Local RTX 4070** (CUDA 12.x): installs latest torch with cu124

In [ ]:
# if IS_RUNPOD:
#     # Pin to CUDA 11.8 — prevents accidental upgrades that break CUDA on RunPod
#     %pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 \
#         --index-url https://download.pytorch.org/whl/cu118 -q
# else:
#     # Local — install latest torch with CUDA 12.4 support (RTX 4070)
#     %pip install torch torchvision \
#         --index-url https://download.pytorch.org/whl/cu124 -q

## 4. Other Python dependencies

In [6]:
%pip install timm pycocoevalcap pycocotools bert-score -q


[notice] A new release of pip is available: 23.3.1 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 5. mamba-ssm CUDA kernel (VMamba speedup)
- **Linux only** — compiles from source, takes **20-40 min**
- **Windows**: not supported — VMamba will use Triton fallback (still works)
- Skip this cell if you don't want to wait

In [ ]:
# if IS_WINDOWS:
#     print("mamba-ssm requires Linux + CUDA. Skipping on Windows.")
#     print("VMamba will use Triton/PyTorch fallback — training still works.")
# else:
#     %pip install mamba-ssm --no-build-isolation

## 6. Cache pretrained weights

In [6]:
import os
if IS_RUNPOD:
    os.environ["HF_HOME"] = "/workspace/hf_cache"
    os.makedirs("/workspace/hf_cache", exist_ok=True)
else:
    cache = os.path.join(PROJECT, ".hf_cache")
    os.environ["HF_HOME"] = cache
    os.makedirs(cache, exist_ok=True)
print("HF_HOME:", os.environ["HF_HOME"])

HF_HOME: /workspace/hf_cache


## 7. Working directory + path

In [7]:
import sys, os
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
print("Working directory:", os.getcwd())

Working directory: /workspace/Pattern_Recog_Project


## 8. Download dataset

In [ ]:
%run src/data/make_data.py
%run src/data/preprocess_annotations.py

## 9. Verify setup

In [10]:
# 1. Eradicate corrupted memory maps and bleeding-edge packages
!pip uninstall -y torch torchvision torchaudio mamba-ssm causal-conv1d quack-kernels
!pip cache purge





Found existing installation: torch 2.1.0+cu118
Uninstalling torch-2.1.0+cu118:
  Successfully uninstalled torch-2.1.0+cu118
Found existing installation: torchvision 0.16.0+cu118
Uninstalling torchvision-0.16.0+cu118:
  Successfully uninstalled torchvision-0.16.0+cu118
Found existing installation: torchaudio 2.1.0+cu118
Uninstalling torchaudio-2.1.0+cu118:
  Successfully uninstalled torchaudio-2.1.0+cu118
Files removed: 232


In [11]:

# 2. Anchor the PyTorch Foundation (Matches RunPod's CUDA 12.0 API constraint)
!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB 3.5 MB/s eta 0:00:000:00:010:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 20.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.5 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.3.1 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [12]:

# 3. Patch the Supporting Cast (Fixes NumPy 2.0 C++ crash & Transformers PyTorch 2.4 boycott)
!pip install "numpy<2" "transformers<4.42.0" --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 11.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 81.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 80.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.6/793.6 kB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 54.5 MB/s eta 0:00:00
   ━━━━

In [12]:

# 4. Surgical Triton Upgrade (Mamba 3 requires `restore_value`, --no-deps protects PyTorch)
!pip install triton==3.6.0 --no-deps


  Using cached triton-3.6.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
Using cached triton-3.6.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (188.1 MB)
  Attempting uninstall: triton
    Found existing installation: triton 2.1.0
    Uninstalling triton-2.1.0:
      Successfully uninstalled triton-2.1.0

[notice] A new release of pip is available: 23.3.1 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [16]:
# 5. Compile Causal-Conv1d 1.4.0 (Pre-dates the PyTorch 2.4 custom_op API break)
!export MAX_JOBS=32
!pip install causal-conv1d==1.4.0 --no-build-isolation --force-reinstall --no-deps


  Using cached causal_conv1d-1.4.0.tar.gz (9.3 kB)
  Preparing metadata (setup.py) ... done
  Created wheel for causal-conv1d: filename=causal_conv1d-1.4.0-cp310-cp310-linux_x86_64.whl size=103612780 sha256=b37d1d2fcbb3d79446fb7f08690a6136da3c92be0ca2aab1629a1ab3d451ce9e
  Stored in directory: /root/.cache/pip/wheels/e3/dd/4c/205f24e151736bd22f5980738dd10a19af6f093b6f4dcab006
Successfully built causal-conv1d

[notice] A new release of pip is available: 23.3.1 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [18]:
!pip install ninja

  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)

[notice] A new release of pip is available: 23.3.1 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [19]:

# 6. Compile Mamba 3 From Source (--no-deps forces it to bind to PyTorch 2.1.2)
!export MAX_JOBS=32
!MAMBA_FORCE_BUILD=TRUE pip install --no-cache-dir --force-reinstall git+https://github.com/state-spaces/mamba.git --no-build-isolation --no-deps

  Cloning https://github.com/state-spaces/mamba.git to /tmp/pip-req-build-1q12zi5k
  Running command git clone --filter=blob:none --quiet https://github.com/state-spaces/mamba.git /tmp/pip-req-build-1q12zi5k
  Resolved https://github.com/state-spaces/mamba.git to commit 316ed6036538405f767782132f76caf342256d33
  Running command git submodule update --init --recursive -q
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.1-cp310-cp310-linux_x86_64.whl size=196305877 sha256=540fb1f93851d4985dc16f07fc67fc2beff8147ded77945abf8ec08fd58ad898
  Stored in directory: /tmp/pip-ephem-wheel-cache-sjfh8fs6/wheels/2c/84/e0/d088f3482f686d0ea9456a883430b1b473fad6a70a408436d6
Successfully built mamba-ssm

[notice] A new release of pip is available: 23.3.1 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [22]:
!pip install einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 1.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mamba-ssm 2.3.1 requires apache-tvm-ffi<=0.1.9, which is not installed.
mamba-ssm 2.3.1 requires quack-kernels>=0.3.4, which is not installed.
mamba-ssm 2.3.1 requires tilelang==0.1.8, which is not installed.

[notice] A new release of pip is available: 23.3.1 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [23]:
!pip install "apache-tvm-ffi<=0.1.9" "quack-kernels==0.3.4" "tilelang==0.1.8"

  Using cached cuda_bindings-13.2.0-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.3 kB)
  Using cached cuda_pathfinder-1.5.3-py3-none-any.whl.metadata (1.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.5/181.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 45.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 60.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 MB 49.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 64.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 77.3 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 MB 46.1 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 61.1 MB/s eta 0:00:00
Using cached cuda_bindings-13.2.0-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (6.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━

In [25]:
import torch, shutil
if not hasattr(torch, 'uint16'):
    torch.uint16 = torch.int16
    torch.uint32 = torch.int32
    torch.uint64 = torch.int64
from src.models.encoder_vmamba import WITH_MAMBA_SSM
import mamba_ssm
from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
import selective_scan_cuda
WITH_MAMBA_SSM = True
print(f"Platform      : {platform.system()}")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU           : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CUDA kernel   : {WITH_MAMBA_SSM}  <- True = fast VMamba selective scan")
print(f"Java (metrics): {shutil.which('java') is not None}  <- True = METEOR/CIDEr will work")
print(f"Flickr8k data : {os.path.exists('data/flickr8k_annotations.json')}  <- True = dataset ready")

/usr/local/lib/python3.10/dist-packages/tvm_ffi/_optional_torch_c_dlpack.py:181: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
We recommend installing via `pip install torch-c-dlpack-ext`
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/tvm_ffi/_optional_torch_c_dlpack.py:181: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
We recommend installing via `pip install torch-c-dlpack-ext`
  warnings.warn(


Platform      : Linux
PyTorch       : 2.1.2+cu118
CUDA available: True
GPU           : NVIDIA GeForce RTX 4090
CUDA kernel   : True  <- True = fast VMamba selective scan
Java (metrics): True  <- True = METEOR/CIDEr will work
Flickr8k data : True  <- True = dataset ready


## 9b. (Optional) Set up SPICE metric
Skip this cell if you don't want SPICE scores during validation. Downloads `spice-1.0.jar` (~5 MB) and Stanford CoreNLP models (~377 MB, one-time). Linux/RunPod only — needs Java (already installed in cell 2).

In [8]:
SPICE_DIR = os.path.join(PROJECT, "src", "models", "spice")
SPICE_JAR = os.path.join(SPICE_DIR, "spice-1.0.jar")
SPICE_LIB = os.path.join(SPICE_DIR, "lib", "stanford-corenlp-3.6.0.jar")

if IS_WINDOWS:
    print("SPICE setup is Linux/RunPod-focused. Skipping on Windows — install Java + jar manually if needed.")
else:
    if not os.path.exists(SPICE_JAR):
        print("Downloading spice-1.0.jar ...")
        os.makedirs(SPICE_DIR, exist_ok=True)
        url = "https://github.com/Aldenhovel/bleu-rouge-meteor-cider-spice-eval4imagecaption/raw/main/spice/spice-1.0.jar"
        subprocess.run(["wget", "-q", "-O", SPICE_JAR, url], check=True)
    print(f"spice-1.0.jar : {os.path.exists(SPICE_JAR)}")

    if not os.path.exists(SPICE_LIB):
        print("Downloading Stanford CoreNLP models (~377 MB, one-time) ...")
        from src.models.spice.get_stanford_models import get_stanford_models
        get_stanford_models()
    print(f"Stanford CoreNLP : {os.path.exists(SPICE_LIB)}")

spice-1.0.jar : True


KeyboardInterrupt: 

## 10. Config — edit before training

In [ ]:
# End-to-end fine-tune: encoder + decoder are both trained.
# (Memory-saver: VMamba encoder + transformer decoder gradient checkpointing is on by default in train.py.)

ENCODER    = "vmamba_small"   # vit_base | vit_small | vmamba_small
DECODER    = "mamba3"    # transformer | mamba | mamba3
DATASET    = "flickr8k"       # flickr8k | mscoco

# VRAM budget for unfrozen vmamba_small + transformer decoder on a 24 GB GPU:
#   BATCH_SIZE 8, GRAD_ACCUM 8  →  effective 64
# Lower BATCH_SIZE if you OOM, raise GRAD_ACCUM to keep the same effective batch.
BATCH_SIZE = 32
GRAD_ACCUM = 8

EPOCHS     = 20
LR         = 4e-4              # lower than frozen-encoder runs; full fine-tune is more sensitive
WORKERS    = 4 if IS_WINDOWS else 12

# Early stopping — halt if CIDEr does not improve for N epochs (0 disables)
PATIENCE   = 3
MIN_DELTA  = 0.01

# Optional metrics
USE_SPICE  = True    # True requires cell 9b to have run successfully

# Resume from a previous checkpoint (e.g. after a RunPod restart). Empty = train from scratch.
# Tip: on RunPod, models/ lives under /workspace/<repo>, which persists if /workspace is a network volume.
RESUME     = ""       # e.g. "models/last.pt"

## 11. Train

In [29]:
!pip uninstall -y quack-kernels

Found existing installation: quack-kernels 0.3.4
Uninstalling quack-kernels-0.3.4:
  Successfully uninstalled quack-kernels-0.3.4


In [11]:
!pip install torch-c-dlpack-ext

  Using cached triton-2.1.0-0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.3 kB)
Using cached triton-2.1.0-0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (89.2 MB)
  Attempting uninstall: triton
    Found existing installation: triton 3.6.0
    Uninstalling triton-3.6.0:
      Successfully uninstalled triton-3.6.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mamba-ssm 2.3.1 requires quack-kernels>=0.3.4, which is not installed.
mamba-ssm 2.3.1 requires triton>=3.5.0, but you have triton 2.1.0 which is incompatible.

[notice] A new release of pip is available: 23.3.1 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip


In [29]:
import subprocess, sys

# Note: --freeze_encoder is intentionally NOT passed — encoder is fine-tuned end-to-end.
cmd = [
    sys.executable, "-u", "src/models/train.py",
    "--dataset",    DATASET,
    "--encoder",    ENCODER,
    "--decoder",    DECODER,
    "--batch_size", str(BATCH_SIZE),
    "--grad_accum", str(GRAD_ACCUM),
    "--lr",         str(LR),
    "--epochs",     str(EPOCHS),
    "--workers",    str(WORKERS),
    "--patience",   str(PATIENCE),
    "--min_delta",  str(MIN_DELTA),
]
if USE_SPICE:
    cmd.append("--spice")
if RESUME:
    cmd += ["--checkpoint", RESUME]

print("Running:", " ".join(cmd))
print(f"Effective batch size: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print("-" * 80)

# Stream output live so you see per-epoch progress in the notebook (subprocess.run buffers).
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    raise
print("-" * 80)
print(f"Exit code: {proc.returncode}")

Running: /usr/bin/python -u src/models/train.py --dataset flickr8k --encoder vmamba_small --decoder mamba3 --batch_size 32 --grad_accum 8 --lr 5e-05 --epochs 20 --workers 12 --patience 3 --min_delta 0.01 --checkpoint models/last.pt
Effective batch size: 32 x 8 = 256
--------------------------------------------------------------------------------
/usr/local/lib/python3.10/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.10/dist-packages/tvm_ffi/_optional_torch_c_dlpack.py:181: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
We recommend installing via `pip install torch-c-dlpack-ext`
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/tvm_ffi/_optional_torch_c_dlpack.py:181: UserWarning: Failed to JIT torch c dlpack ex